# Malayalam ASR eval — accuracy per source + CPU speed on 2 cores

Runs every model in [malayalam-asr-eval](https://github.com/cksajil/malayalam-asr-eval) on the
**231-clip leakage-free subset** (50 per source, seed 42 — the same clips as the published v1/v2 numbers),
one model per process, pinned to **2 physical cores**, batch 1, greedy decoding.

**Order:** `whisper-v2` runs first as a sanity check. It must land near **31.2% WER / 4.0% CER**; if it
doesn't, the notebook stops, because nothing after it would be comparable.

**Kaggle setup**
- Accelerator → **None** (CPU). Internet → **On**.
- Code: attach the Kaggle dataset that contains the `malayalam-asr-eval` folder; if none is attached, the repo is cloned from GitHub.
- Add-ons → Secrets → `HF_TOKEN` (needed for IndicConformer, which is gated: accept its conditions on
  huggingface.co/ai4bharat/indic-conformer-600m-multilingual first). Without it that one model is skipped.
- Runtime: roughly 4–5 h for the default plan (vrclc medium is the slowest, so it runs last).
  Finished models are skipped on re-run, and an interrupted model resumes from its last checkpoint.

In [ ]:
# ---------------- CONFIG ----------------
REPO_URL   = "https://github.com/cksajil/malayalam-asr-eval"
BRANCH     = "main"         # use the PR branch name here until it is merged
CORES      = 2              # physical cores to pin each run to
PLAN       = ["whisper-v2",          # sanity check, must be first
              "fw-v2-int8",          # CTranslate2 int8 of v2
              "indicconformer-ctc",  # CTC, non-autoregressive
              "mms-1b-all",          # wav2vec2 CTC (MMS, Malayalam adapter)
              "kavya-small",         # competitor
              "vrclc-medium"]        # competitor, slowest
OPTIONAL   = []             # e.g. ["indicconformer-rnnt", "xlsr-ml-shawn", "whisper-v1"]
RUN_FULL   = False          # also score the full 2,684-clip clean set (~88% Shrutilipi; ~10x longer)
SANITY_TOL = {"WER": 1.0, "CER": 0.5}   # percentage points vs published v2
# Re-runs: attach an earlier results.zip as a Kaggle dataset and finished models are reused.
# Models listed here are run again anyway (whisper-v2 first, so the sanity check and the
# speed comparison come from this session's CPU).
RERUN      = ["whisper-v2", "fw-v2-int8"]
WORK       = "/kaggle/working"

In [ ]:
import os, sys, json, subprocess, shutil, time
import glob
os.chdir(WORK)
REPO = f"{WORK}/malayalam-asr-eval"
# Code source: an attached Kaggle dataset containing the repo (if any), otherwise GitHub.
import zipfile

def find_attached_code():
    """Folder containing the eval code in /kaggle/input: extracted, or still inside a .zip."""
    for p in sorted(glob.glob("/kaggle/input/**/mlasr_eval/runner.py", recursive=True)):
        root = os.path.dirname(os.path.dirname(p))
        if os.path.exists(f"{root}/pyproject.toml"):
            return root
    for z in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                hits = [n for n in zf.namelist() if n.endswith("mlasr_eval/runner.py")]
                if hits:
                    out = f"{WORK}/_code_unzipped"
                    shutil.rmtree(out, ignore_errors=True)
                    zf.extractall(out)
                    return os.path.join(out, hits[0][: -len("mlasr_eval/runner.py")])
        except zipfile.BadZipFile:
            pass
    return None

if os.path.isdir(REPO) and not os.path.exists(f"{REPO}/mlasr_eval/runner.py"):
    shutil.rmtree(REPO)              # leftover from a failed attempt (e.g. a clone without the code)
if not os.path.isdir(REPO):
    src = find_attached_code()
    if src:
        shutil.copytree(src, REPO)   # /kaggle/input is read-only; the install needs a writable copy
        print("code from attached dataset:", src)
    else:
        print("No eval code found in the attached datasets. What is attached:")
        for p in sorted(glob.glob("/kaggle/input/*/*")) + sorted(glob.glob("/kaggle/input/*/*/*"))[:30]:
            print("  ", p)
        print("Trying GitHub:", REPO_URL, BRANCH)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=False)
if not os.path.exists(f"{REPO}/mlasr_eval/runner.py"):
    shutil.rmtree(REPO, ignore_errors=True)
    raise SystemExit("Eval code not found. Attach the 'malayalam-asr-eval-code' dataset "
                     "(Add Input), or set BRANCH to a GitHub branch that has the code.")
# Every model runs in a fresh subprocess, so packages installed here are picked up without a kernel restart.
subprocess.run(f"pip install -q -e '{REPO}[all]'", shell=True, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret - gated models (IndicConformer) will be skipped:", type(e).__name__)
HAS_TOKEN = bool(os.environ.get("HF_TOKEN"))

print(subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread|Core|Socket|Flags' | cut -c1-160",
                     shell=True, capture_output=True, text=True).stdout)
import importlib.metadata as md_
for p in ["torch", "transformers", "ctranslate2", "faster-whisper", "onnxruntime", "jiwer", "pandas"]:
    try: print(f"{p:16s} {md_.version(p)}")
    except Exception: print(f"{p:16s} -")

## 1. Evaluation data (download, filter to clean rows, sample, decode once)

In [ ]:
def cli(*args, log=None):
    cmd = [sys.executable, "-u", "-m", "mlasr_eval", *args]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not any(k in line for k in ("Warning", "warn(", "it/s]")):
            print(line, end="", flush=True)
    p.wait()
    if log:
        open(log, "w").write("".join(lines))
    return p.returncode, "".join(lines)

rc, _ = cli("prepare", "--subset", "balanced231", "--out", f"{WORK}/eval_data")
assert rc == 0, "data preparation failed"
DATA = f"{WORK}/eval_data/balanced231.pkl"
RESULTS = f"{WORK}/results"

prior = glob.glob("/kaggle/input/**/balanced231/*/summary.json", recursive=True)
if not prior:   # results dataset stored as a .zip rather than extracted
    for z in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.startswith("balanced231/") and n.endswith("summary.json") for n in zf.namelist()):
                    zf.extractall(f"{WORK}/_prior_results")
                    prior = glob.glob(f"{WORK}/_prior_results/balanced231/*/summary.json")
                    break
        except zipfile.BadZipFile:
            pass
if prior and not os.path.isdir(f"{RESULTS}/balanced231"):
    root = os.path.dirname(os.path.dirname(os.path.dirname(prior[0])))
    shutil.copytree(root, RESULTS, ignore=shutil.ignore_patterns("*.log"))
    print("reusing earlier results from", root, ":", sorted(os.listdir(f"{RESULTS}/balanced231")))
for name in RERUN:
    if os.path.isdir(f"{RESULTS}/balanced231/{name}"):
        shutil.rmtree(f"{RESULTS}/balanced231/{name}")
        print("will re-run", name)

## 2. Sanity check: v2 must reproduce 31.2% WER / 4.0% CER

In [ ]:
def summary_path(name, subset="balanced231"):
    return f"{RESULTS}/{subset}/{name}/summary.json"

def run(name, data=DATA):
    if os.path.exists(summary_path(name, os.path.splitext(os.path.basename(data))[0])):
        print(f"[{name}] already done, skipping"); return 0
    t0 = time.time()
    os.makedirs(RESULTS, exist_ok=True)
    rc, out = cli("run", name, "--data", data, "--results", RESULTS, "--cores", str(CORES),
                  log=f"{RESULTS}/{name}.log")
    print(f"[{name}] exit {rc} after {(time.time()-t0)/60:.1f} min\n")
    return rc

assert run("whisper-v2") == 0, "whisper-v2 run failed - see results/whisper-v2.log"
s = json.load(open(summary_path("whisper-v2")))
c = s["sanity"]
print(f"v2 WER {100*s['overall']['WER']:.1f}% (expected 31.2, {c['delta_WER_pts']:+.1f} pts)  "
      f"CER {100*s['overall']['CER']:.1f}% (expected 4.0, {c['delta_CER_pts']:+.1f} pts)  "
      f"near-cap clips {s['overall']['near_cap']}  RTF {s['overall']['RTF']:.2f}")
if abs(c["delta_WER_pts"]) > SANITY_TOL["WER"] or abs(c["delta_CER_pts"]) > SANITY_TOL["CER"]:
    raise SystemExit("Sanity check FAILED - the setup does not match the published eval. "
                     "Check the transformers version (4.46.3), max_new_tokens (444) and the clip manifest.")
print("Sanity check passed - setup matches the published numbers.")

## 3. All other models

In [ ]:
sys.path.insert(0, REPO)   # the editable install isn't visible to this kernel until a restart
from mlasr_eval.models import MODELS
status = {}
for name in PLAN[1:] + OPTIONAL:
    if MODELS[name].get("gated") and not HAS_TOKEN:
        status[name] = "skipped (needs HF_TOKEN)"; print(f"[{name}] {status[name]}"); continue
    rc = run(name)
    status[name] = "ok" if rc == 0 else f"FAILED (exit {rc}) - see results/{name}.log"
status

## 4. Leaderboard

In [ ]:
from IPython.display import Markdown, display
from mlasr_eval.report import write_report
display(Markdown(open(write_report(RESULTS, "balanced231")).read()))

## 5. Optional: full 2,684-clip clean set

In [ ]:
if RUN_FULL:
    rc, _ = cli("prepare", "--subset", "full", "--out", f"{WORK}/eval_data")
    for name in PLAN + OPTIONAL:
        if MODELS[name].get("gated") and not HAS_TOKEN:
            continue
        run(name, data=f"{WORK}/eval_data/full.pkl")
    display(Markdown(open(write_report(RESULTS, "full")).read()))

## 6. Package results (download `results.zip`, or commit `results/` to the repo)

In [ ]:
shutil.copy(f"{WORK}/eval_data/balanced231_manifest.csv", f"{RESULTS}/balanced231/manifest.csv")
shutil.make_archive(f"{WORK}/results", "zip", RESULTS)
print(sorted(os.listdir(f"{RESULTS}/balanced231")))